In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

import torch
import yaml
import numpy as np
import glob
from collections import Counter
from ultralytics import YOLO
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

PROJECT_ROOT = '/mnt/Data/AKIB/YOLO-OD-IM'
DATA_YAML    = os.path.join(PROJECT_ROOT, 'dataset', 'data_abs.yaml')

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)
class_names = cfg['names']
nc = cfg['nc']

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"Classes: {nc}")

## 2.1 Analyze Class Imbalance

Before training, let's understand the class distribution to justify our optimization choices.

In [ ]:
# ── Analyze class distribution ──
train_label_dir = os.path.join(PROJECT_ROOT, 'dataset', 'train', 'labels')
class_counter = Counter()

for lf in glob.glob(os.path.join(train_label_dir, '*.txt')):
    with open(lf) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                class_counter[int(parts[0])] += 1

# Class frequency array
counts = np.array([class_counter.get(i, 0) for i in range(nc)])

fig, ax = plt.subplots(figsize=(16, 6))
sorted_idx = np.argsort(counts)[::-1]
colors = ['#e74c3c' if counts[i] < 10 else '#f39c12' if counts[i] < 50 else '#2ecc71' for i in sorted_idx]
ax.bar(range(nc), counts[sorted_idx], color=colors)
ax.set_xticks(range(nc))
ax.set_xticklabels([class_names[i] for i in sorted_idx], rotation=90, fontsize=5)
ax.set_ylabel('Instance Count')
ax.set_title('Training Set Class Distribution (Red: <10, Orange: <50, Green: ≥50)')
ax.set_yscale('log')
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\nImbalance ratio: {counts[counts>0].max()}/{counts[counts>0].min()} = {counts[counts>0].max()/counts[counts>0].min():.1f}x")
print(f"Classes with 0 samples: {(counts == 0).sum()}")
print(f"Classes with <10 samples: {(counts < 10).sum()}")
print(f"Classes with <50 samples: {(counts < 50).sum()}")

## 2.2 Train YOLOv11m – Optimized for Recall

Key settings:
- **`fl_gamma=2.0`**: Focal loss to down-weight easy (frequent) class examples
- **`imgsz=1280`**: Larger resolution for small products on shelf
- **`mosaic=1.0, mixup=0.15, copy_paste=0.1`**: Aggressive augmentation
- **`cos_lr=True`**: Cosine learning rate for better convergence
- **`close_mosaic=15`**: Disable mosaic in last 15 epochs for fine-tuning

In [ ]:
# ── Train YOLOv11m with recall-optimized settings ──
model_m = YOLO('yolo11m.pt')

results_m = model_m.train(
    data=DATA_YAML,
    epochs=150,
    imgsz=1280,              # larger resolution for small shelf products
    batch=8,                 # reduced batch for 1280 on 24GB VRAM
    device=0,
    project=os.path.join(PROJECT_ROOT, 'runs'),
    name='optimized_yolo11m_1280',
    exist_ok=True,
    
    # ── Loss & Optimization ──
    fl_gamma=2.0,            # focal loss for class imbalance
    cos_lr=True,             # cosine annealing LR
    lr0=0.01,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=5,
    
    # ── Augmentation ──
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,
    close_mosaic=15,
    degrees=10.0,
    translate=0.2,
    scale=0.5,
    shear=5.0,
    flipud=0.5,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    erasing=0.1,
    
    # ── Training strategy ──
    patience=30,
    save=True,
    save_period=25,
    plots=True,
    verbose=True,
)

## 2.3 Train YOLOv11l – Maximum Capacity

Train the large model for potential ensemble use later.

In [ ]:
# ── Train YOLOv11l with same optimized settings ──
model_l = YOLO('yolo11l.pt')

results_l = model_l.train(
    data=DATA_YAML,
    epochs=150,
    imgsz=1280,
    batch=4,                 # smaller batch for larger model
    device=0,
    project=os.path.join(PROJECT_ROOT, 'runs'),
    name='optimized_yolo11l_1280',
    exist_ok=True,
    
    # ── Same loss & optimization ──
    fl_gamma=2.0,
    cos_lr=True,
    lr0=0.01,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=5,
    
    # ── Same augmentation ──
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,
    close_mosaic=15,
    degrees=10.0,
    translate=0.2,
    scale=0.5,
    shear=5.0,
    flipud=0.5,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    erasing=0.1,
    
    patience=30,
    save=True,
    save_period=25,
    plots=True,
    verbose=True,
)

## 2.4 Evaluate Optimized Models

In [ ]:
# ── Compare all models ──
model_configs = [
    ('YOLOv11n (baseline)', os.path.join(PROJECT_ROOT, 'runs', 'baseline_yolo11n', 'weights', 'best.pt')),
    ('YOLOv11m (optimized)', os.path.join(PROJECT_ROOT, 'runs', 'optimized_yolo11m_1280', 'weights', 'best.pt')),
    ('YOLOv11l (optimized)', os.path.join(PROJECT_ROOT, 'runs', 'optimized_yolo11l_1280', 'weights', 'best.pt')),
]

comparison = []
for name, path in model_configs:
    if os.path.exists(path):
        model = YOLO(path)
        # Test with optimized conf=0.15
        m = model.val(data=DATA_YAML, split='test', device=0, conf=0.15, iou=0.6, verbose=False)
        comparison.append({
            'name': name,
            'precision': m.box.mp,
            'recall': m.box.mr,
            'map50': m.box.map50,
            'map': m.box.map,
        })
        print(f"{name:30s} → P={m.box.mp:.4f}, R={m.box.mr:.4f}, mAP50={m.box.map50:.4f}")
    else:
        print(f"{name:30s} → weights not found at {path}")

# Plot comparison
if len(comparison) > 1:
    fig, ax = plt.subplots(figsize=(10, 6))
    names = [c['name'] for c in comparison]
    x = range(len(names))
    w = 0.25
    
    ax.bar([i-w for i in x], [c['precision'] for c in comparison], w, label='Precision', color='coral')
    ax.bar([i   for i in x], [c['recall']    for c in comparison], w, label='Recall', color='steelblue')
    ax.bar([i+w for i in x], [c['map50']     for c in comparison], w, label='mAP@50', color='mediumseagreen')
    
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=15)
    ax.set_ylabel('Metric Value')
    ax.set_title('Model Comparison (conf=0.15, iou=0.6)')
    ax.legend()
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for i, c in enumerate(comparison):
        ax.text(i-w, c['precision']+0.02, f"{c['precision']:.3f}", ha='center', fontsize=7)
        ax.text(i, c['recall']+0.02, f"{c['recall']:.3f}", ha='center', fontsize=7)
        ax.text(i+w, c['map50']+0.02, f"{c['map50']:.3f}", ha='center', fontsize=7)
    
    plt.tight_layout()
    plt.savefig(os.path.join(PROJECT_ROOT, 'model_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── Per-class improvement analysis ──
# Compare per-class recall between baseline and best optimized model
baseline_path = os.path.join(PROJECT_ROOT, 'runs', 'baseline_yolo11n', 'weights', 'best.pt')
best_opt_path = os.path.join(PROJECT_ROOT, 'runs', 'optimized_yolo11m_1280', 'weights', 'best.pt')

if os.path.exists(baseline_path) and os.path.exists(best_opt_path):
    m_base = YOLO(baseline_path).val(data=DATA_YAML, split='test', device=0, conf=0.15, iou=0.6, verbose=False)
    m_opt  = YOLO(best_opt_path).val(data=DATA_YAML, split='test', device=0, conf=0.15, iou=0.6, verbose=False)
    
    recall_diff = m_opt.box.r - m_base.box.r
    
    fig, ax = plt.subplots(figsize=(16, 6))
    sorted_idx = np.argsort(recall_diff)
    colors = ['#e74c3c' if d < 0 else '#2ecc71' for d in recall_diff[sorted_idx]]
    ax.bar(range(nc), recall_diff[sorted_idx], color=colors)
    ax.set_xticks(range(nc))
    ax.set_xticklabels([class_names[i] for i in sorted_idx], rotation=90, fontsize=5)
    ax.set_ylabel('Recall Difference (Optimized - Baseline)')
    ax.set_title('Per-Class Recall Improvement (Green=improved, Red=degraded)')
    ax.axhline(y=0, color='black', linewidth=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(PROJECT_ROOT, 'recall_improvement.png'), dpi=150, bbox_inches='tight')
    plt.show()
    
    improved = (recall_diff > 0).sum()
    degraded = (recall_diff < 0).sum()
    print(f"\nClasses improved: {improved}/{nc}")
    print(f"Classes degraded: {degraded}/{nc}")
    print(f"Mean recall improvement: {recall_diff.mean():.4f}")